# Apt 305 — the canonical trajectory on the **clean** weather file

Re-runs the established ten-state trajectory in methodology order. **No engine
logic changes. Only the weather input changes.**

## Why

The previous canonical run used `AUS_VIC_Melbourne.RO.948680_TMYx.2011-2025.epw`
(Melbourne Regional Office, WMO 948680). It is site-correct — 0.008° from the
building, closer than the replacement — and still unusable. Four whole calendar
months of its wind column read **exactly 0.0 m/s for every hour**:

| | |
| --- | --- |
| Dead-calm months | January, March, July, September |
| Hours | 2 952 — **33.7 % of the year** |
| Cause | the station's record ends in 2014; the TMYx composite zero-filled the gap |
| Why it is silent | the EPW missing-value code for wind (999) appears nowhere in the file, so nothing marks those hours as absent and the engine reads them as genuine calm |

Correction **C2** replaces the ISO fixed external convective coefficient with
`h_ce = 4v + 4`. At v = 0 that gives 4 W/(m²·K) against the ISO constant's 20 —
a five-fold *weakening* of the external film on this apartment's one exposed
surface (a west wall, solar absorptance 0.75). Weaken it on a sunny afternoon and
the wall sheds much less of the absorbed solar to the air, its sol-air driving
temperature climbs, and heat is conducted inward instead. The earlier wind
diagnostic traced **96 % of the C2 cooling increase to those fabricated
zero-wind hours** and returned verdict **(c)**: not explained by real wind.

So the 9.69 kWh/m²·yr headline was partly built on bad data.

## The replacement

`AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw` — Essendon Fields,
WMO 958660, ~8 km NW of the Carlton site, complete continuous record. 8 760 rows,
**0 missing wind values, no dead-calm month**, annual mean 4.84 m/s, 1.58 % of
hours exactly zero, **59.8 % above the 4 m/s pivot**. Genuinely windier than the
broken file implied, so C2 now has a real, physically-sound effect.

## What this notebook does

1. Places and **screens** the weather file — aborts on a dead-calm month.
2. Re-runs the ten-state trajectory and reproduces every acceptance artefact.
3. Reports each invariant explicitly: V2 residual < 5 % on every state, ADJ
   transmission in the inventory, latent gating, HEAD invariance.
4. Re-runs the wind diagnostic, closing the earlier verdict (c).
5. Runs the full regression suite.

Runtime is roughly **25–40 minutes** on a standard Colab CPU runtime — the
trajectory builds and measures eleven engine trees. No GPU, no EnergyPlus.

## 1 · Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO   = Path('/content/AIB')
BRANCH = 'claude/aib-canonical-clean-weather-0fr0mp'   # the clean-weather branch

if not REPO.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/samiraghafarigousheh-sys/aib.git', str(REPO)],
                   check=True)
os.chdir(REPO)

# EVERY branch, not just the default one. The trajectory cherry-picks ten states
# onto the vendored baseline and two test modules build worktrees from the
# historical fix branches; without the full refspec those tests skip and the
# trajectory cannot resolve its commits.
subprocess.run(['git', 'fetch', 'origin', '+refs/heads/*:refs/remotes/origin/*'],
               check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=False)

# Cherry-picking needs a committer identity. A fresh Colab runtime has none and
# git fails with "Please tell me who you are", which points nowhere near the
# harness.
subprocess.run(['git', 'config', 'user.email', 'harness@localhost'], check=True)
subprocess.run(['git', 'config', 'user.name',  'aib-harness'], check=True)

# Colab already ships numpy, pandas, matplotlib, plotly, scipy, scikit-learn,
# pytz, requests, tqdm and pytest. These five are the gaps.
#
# pyecharts is only a rendering library and the engine imports fine without it on
# this branch — but the regression suite drives worktrees of the HISTORICAL
# branches, whose package still imports it eagerly. Without it those tests error
# at fixture setup rather than failing honestly.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pvlib', 'timezonefinder', 'holidays', 'workalendar',
                'pyecharts'], check=True)

print(subprocess.run(['git', 'log', '--oneline', '-3'],
                     capture_output=True, text=True).stdout)
print('EPWs present:')
for p in sorted(Path('weather_cache').glob('*.epw')):
    print('  ', p.name)

## 2 · The weather file, screened before anything is computed

Site-correctness was never sufficient — the RO file passed every site check and
was still unusable. `tools/diagnostics/weather_integrity.py` reads the wind
column directly (pure stdlib, no engine, no pandas) and reports the resolved
**absolute** path, the station header, and the three numbers that matter. Both
files are screened here, so the contrast is on the page rather than asserted.

In [ ]:
subprocess.run([sys.executable, 'tools/diagnostics/weather_integrity.py',
                'weather_cache/AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw',
                'weather_cache/AUS_VIC_Melbourne.RO.948680_TMYx.2011-2025.epw',
                '--no-assert',
                '--json', 'results/diagnostics/weather_integrity_essendon_vs_ro.json'])

### The hard stop

`--no-assert` above was for reporting. Without it the screen **aborts**, and it
is wired into weather resolution, the trajectory harness and the wind
diagnostic. The cell below proves the guard is live by pointing it at the file it
exists to catch.

In [ ]:
sys.path.insert(0, str(REPO / 'examples'))
from weather_melbourne import resolve, CANONICAL_EPW, SUPERSEDED_EPW

print('CANONICAL_EPW  =', CANONICAL_EPW)
print('SUPERSEDED_EPW =', SUPERSEDED_EPW, '\n')

# What the case study resolves to with no --weather argument.
src, path, label = resolve(None, 'auto')
print('resolved ->', path, '\n')
assert 'Essendon' in path, 'the case study is not on the Essendon file'

# And what happens if the superseded file is passed explicitly.
print('--- passing the RO file explicitly ---')
try:
    resolve(f'weather_cache/{SUPERSEDED_EPW}', 'epw')
    print('\n!! NOT REFUSED — the guard is not working')
except Exception as exc:
    print('\nREFUSED, correctly:')
    print('   ', str(exc).strip().splitlines()[1].strip())

## 3 · Run the trajectory

Ten states, methodology order, each the previous state plus **exactly one**
correction cherry-picked onto the unmodified vendored baseline (`2e6e910`):

    Baseline → C1 dynamic window → C2 wind-dependent h_ce → Ventilation
             → Latent → Internal gains → Conditioned zones → Ground contact
             → Hemisphere → Closure fixes (HEAD)

Every state is measured with the **same** closure-capable instrument — the
closure commits are cherry-picked onto each one — which is the only thing that
makes them comparable. An eleventh run measures HEAD directly, as the invariance
reference.

Two harness details worth knowing before you read the output:

* `--closure-base 978db37` is **pinned**, not `origin/main`. PR #16 merged three
  of the four closure commits into `main`, so the old default now resolves to a
  single commit; nine states would be measured with an instrument the tenth does
  not use, the residuals would blow out, and it would look like a physics
  finding. The tool asserts the resolved set and refuses a short one.
* HEAD invariance is checked against **HEAD run on this same weather**, not
  against a stored constant. A stored constant asserts "reproduces the number we
  published on the RO file", which is false by design here and says nothing
  about whether the corrections are separable.

Watch the `resid=` column and the `PASS` flags. This takes ~20–30 minutes.

In [ ]:
EPW = 'weather_cache/AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw'

proc = subprocess.Popen(
    [sys.executable, 'tools/diagnostics/canonical_trajectory.py',
     '--weather', EPW,
     '--outdir', 'results/au_canonical_essendon',
     '--expect-weather', 'Essendon'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:          # stream, rather than going quiet for half an hour
    print(line, end='')
proc.wait()

print('\nexit code:', proc.returncode)
print('  0 = gate passed, HEAD invariant, engine tree identical, ADJ in inventory, latent gated')
print('  2 = a state failed the 5 % V2 residual gate')
print('  3 = HEAD is not invariant — a correction is not cleanly separable')
print('  4 = the final engine tree differs from HEAD')
print('  5 = the transmission inventory or the latent gate failed')

## 4 · The trajectory table

In [ ]:
import json
import pandas as pd

RAW  = Path('results/au_canonical_essendon/trajectory_raw.json')
blob = json.loads(RAW.read_text())
results, meta, head = blob['results'], blob['meta'], blob['head']
AREA = float(next(iter(results.values()))['config_B']['net_floor_area_m2'])

def B(state):
    return results[state]['config_B']

traj = pd.DataFrame([{
    'State': s,
    'Sensible heating (kWh)': B(s)['Q_H_sensible_kWh'],
    'Sensible cooling (kWh)': B(s)['Q_C_sensible_kWh'],
    'Latent cooling, gated (kWh)': B(s)['Q_C_latent_kWh'],
    'Latent cooling, ungated (kWh)': B(s)['Q_C_latent_ungated_kWh'],
    'Latent heating (kWh)': B(s)['Q_H_latent_kWh'],
    'Sens + gated latent (kWh)': (B(s)['Q_H_sensible_kWh'] + B(s)['Q_C_sensible_kWh']
                                  + B(s)['Q_C_latent_kWh']),
    'Total (kWh)': B(s)['Q_need_total_kWh'],
    'Total (kWh/m²)': B(s)['Q_need_total_kWh_per_sqm'],
    'V2 residual (kWh)': B(s)['sankey']['residual_kWh'],
    'V2 residual (%)': B(s)['sankey']['residual_pct'],
    'Gate': 'PASS' if abs(B(s)['sankey']['residual_pct']) < 5.0 else 'FAIL',
    'Tr items': B(s)['sankey']['n_transmission_items'],
} for s in results])

display(traj.style.format({
    **{c: '{:,.2f}' for c in traj.columns if traj[c].dtype.kind == 'f'},
    'Latent heating (kWh)': '{:,.4f}',
    'V2 residual (kWh)': '{:+,.2f}',
    'V2 residual (%)': '{:+.2f} %',
}).hide(axis='index'))

print('`Latent cooling, ungated` is the diagnostic contrast column only — the zone')
print('moisture balance before the plant-on gate. It is never part of a total.')
print()
print('`Total (kWh)` is the engine\'s own total and additionally carries latent')
print('HEATING, which is ~153 kWh of phantom humidification on the states before')
print('the latent fix and 0.00 after it. That is why the two total columns diverge')
print('early and coincide at the canonical state.')

## 5 · The new canonical headline, and what it supersedes

Same engine, same building, same commit. Only the weather file differs.

In [ ]:
final = traj.iloc[-1]

# The superseded run, read from what was actually published rather than retyped.
prior_csv = Path('results/au_canonical/comparison.csv')
prior = pd.read_csv(prior_csv).iloc[-1] if prior_csv.is_file() else None

print(f"NEW CANONICAL HEADLINE — {Path(meta['weather']).name}\n")
print(f"  {final['Sensible heating (kWh)']:.2f} kWh sensible heating")
print(f"+ {final['Sensible cooling (kWh)']:.2f} kWh sensible cooling")
print(f"+ {final['Latent cooling, gated (kWh)']:.2f} kWh gated latent")
print(f"= {final['Sens + gated latent (kWh)']:.2f} kWh")
print(f"= {final['Total (kWh/m²)']:.2f} kWh/m²·yr   over {AREA:.0f} m²\n")

if prior is not None:
    cmp = pd.DataFrame([
        ('Sensible heating (kWh)',   prior['Sensible heating (kWh)'],       final['Sensible heating (kWh)']),
        ('Sensible cooling (kWh)',   prior['Sensible cooling (kWh)'],       final['Sensible cooling (kWh)']),
        ('Gated latent (kWh)',       prior['Latent cooling, gated (kWh)'],  final['Latent cooling, gated (kWh)']),
        ('Total (kWh)',              prior['Total (kWh)'],                  final['Total (kWh)']),
        ('Total (kWh/m²·yr)',        prior['Total (kWh/m²)'],               final['Total (kWh/m²)']),
    ], columns=['Metric', 'Prior — RO, corrupt wind', 'This run — Essendon'])
    cmp['Δ']   = cmp['This run — Essendon'] - cmp['Prior — RO, corrupt wind']
    cmp['Δ %'] = 100 * cmp['Δ'] / cmp['Prior — RO, corrupt wind']
    display(cmp.style.format({'Prior — RO, corrupt wind': '{:,.2f}',
                              'This run — Essendon': '{:,.2f}',
                              'Δ': '{:+,.2f}', 'Δ %': '{:+.1f} %'}).hide(axis='index'))
    print('The previous headline was 9.69 kWh/m²·yr, on the corrupt wind column.')
else:
    print('results/au_canonical/comparison.csv not present — contrast omitted rather than guessed.')

### The C2 step is where the weather change acts

Expected, and it is a **reversal of sign**. With 59.8 % of hours above the pivot,
`4v + 4` now sits *above* the ISO fixed 20 W/(m²·K) for most of the year instead
of collapsing to a fifth of it. A stronger external film sheds more of the
absorbed solar from the west wall back to the air, so the correction *reduces*
cooling rather than manufacturing it.

In [ ]:
c1 = B('+C1 dynamic window')
c2 = B('+C2 wind-dependent h_ce')
dC = c2['Q_C_sensible_kWh'] - c1['Q_C_sensible_kWh']
dH = c2['Q_H_sensible_kWh'] - c1['Q_H_sensible_kWh']

print(f"C2 step, sensible cooling : {c1['Q_C_sensible_kWh']:8.2f} -> "
      f"{c2['Q_C_sensible_kWh']:8.2f} kWh   ({dC:+.2f})")
print(f"C2 step, sensible heating : {c1['Q_H_sensible_kWh']:8.2f} -> "
      f"{c2['Q_H_sensible_kWh']:8.2f} kWh   ({dH:+.2f})")
print()
print('On the superseded RO file the same step moved cooling +119.58 kWh, and the')
print('wind diagnostic traced 96 % of that to hours reading exactly 0.0 m/s.')
print('Section 9 isolates C2 on this file with a one-switch controlled experiment.')

## 6 · The invariants

Each reported explicitly rather than folded into a single pass/fail. Any one of
these failing means no kWh/m² headline may be quoted from the run.

In [ ]:
V2_TOL, REINT_TOL, EXPECTED_ITEMS = 5.0, 0.001, 7

# --- V2 residual ---------------------------------------------------------
worst = traj.loc[traj['V2 residual (%)'].abs().idxmax()]
gate_ok = (traj['V2 residual (%)'].abs() < V2_TOL).all()
print(f"V2 residual < {V2_TOL:.0f} % on every state          : "
      f"{'PASS' if gate_ok else 'FAIL'}"
      f"   (worst {worst['V2 residual (%)']:+.2f} % at {worst['State']})")

# --- ADJ transmission in the inventory -----------------------------------
items_ok = (traj['Tr items'] == EXPECTED_ITEMS).all()
reint    = pd.Series({s: B(s)['sankey']['transmission_rel_diff'] for s in results})
reint_ok = (reint <= REINT_TOL).all()
print(f"{EXPECTED_ITEMS} transmission line items, every state   : "
      f"{'PASS' if items_ok else 'FAIL'}")
print(f"independent re-integration within 0.1 %  : "
      f"{'PASS' if reint_ok else 'FAIL'}   (worst {100 * reint.max():.4f} %)")

# --- latent gate ---------------------------------------------------------
off  = pd.Series({s: B(s).get('latent_kWh_with_cooling_off') or 0.0 for s in results})
heat = pd.Series({s: B(s).get('latent_kWh_while_heating')   or 0.0 for s in results})
lat_ok = off.abs().max() <= 1e-9 and heat.abs().max() <= 1e-9
print(f"latent charged only with cooling on      : {'PASS' if lat_ok else 'FAIL'}"
      f"   (plant off {off.abs().max():.2e} kWh, while heating {heat.abs().max():.2e} kWh)")
print(f"latent heating at the canonical state    : "
      f"{final['Latent heating (kWh)']:.4f} kWh")

# --- southern-hemisphere phase -------------------------------------------
monthly = B(list(results)[-1]).get('monthly_latent_cooling_kWh')
if monthly:
    summer, winter = monthly[0] + monthly[1] + monthly[11], sum(monthly[5:8])
    print(f"southern-hemisphere phase                : "
          f"{'PASS' if summer > winter else 'FAIL'}"
          f"   (Dec-Feb {summer:.2f} kWh vs Jun-Aug {winter:.2f} kWh)")

# --- HEAD invariance ------------------------------------------------------
KEYS = ['Q_H_sensible_kWh', 'Q_C_sensible_kWh', 'Q_C_latent_kWh',
        'Q_need_total_kWh_per_sqm']
last = B(list(results)[-1])
inv = pd.DataFrame([{'Metric': k, 'HEAD, run directly': head['config_B'][k],
                     "Trajectory's final state": last[k],
                     'Δ': abs(head['config_B'][k] - last[k])} for k in KEYS])
head_ok = (inv['Δ'] <= 0.01).all()
tree_ok = blob['engine_tree_check']['identical']
print(f"HEAD invariant under the reordering      : {'PASS' if head_ok else 'FAIL'}")
print(f"final engine tree identical to HEAD      : {'PASS' if tree_ok else 'FAIL'}")
display(inv.style.format({'HEAD, run directly': '{:,.4f}',
                          "Trajectory's final state": '{:,.4f}',
                          'Δ': '{:.2e}'}).hide(axis='index'))

assert all([gate_ok, items_ok, reint_ok, lat_ok, head_ok, tree_ok]), \
    'an invariant failed — no headline may be quoted from this run'
print('\nALL INVARIANTS HOLD.')

### The ADJ transmission inventory, per state

The five party surfaces are 75.10 m², **88.6 % of the envelope UA**, and were
absent from both sides of the balance before the closure fixes. The last two
columns come from different code paths — the reported figure from the in-loop
accumulator, the independent one re-integrated from the hourly frame afterwards.

In [ ]:
adj = pd.DataFrame([{
    'State': s,
    'Line items': B(s)['sankey']['n_transmission_items'],
    'ADJ loss (kWh)': B(s).get('Q_tr_adjacent_loss_kWh'),
    'ADJ gain (kWh)': B(s).get('Q_tr_adjacent_gain_kWh'),
    'Reported Σ (kWh)': B(s)['sankey']['transmission_reported_kWh'],
    'Independent Σ (kWh)': B(s)['sankey']['transmission_independent_kWh'],
    'Δ %': 100 * B(s)['sankey']['transmission_rel_diff'],
} for s in results])
display(adj.style.format({'ADJ loss (kWh)': '{:,.2f}', 'ADJ gain (kWh)': '{:,.2f}',
                          'Reported Σ (kWh)': '{:,.2f}',
                          'Independent Σ (kWh)': '{:,.2f}',
                          'Δ %': '{:.4f} %'}).hide(axis='index'))

print('Final-state transmission line items (kWh):')
for name, kwh in sorted(B(list(results)[-1])['sankey']['transmission_items_kWh'].items(),
                        key=lambda kv: -kv[1]):
    print(f'  {name:<52} {kwh:10,.2f}')

### The latent gate, per state

In [ ]:
lat = pd.DataFrame([{
    'State': s,
    'Steps': B(s).get('n_steps'),
    'Cooling on': B(s).get('n_steps_cooling_on'),
    'Heating on': B(s).get('n_steps_heating_on'),
    'Latent charged': B(s).get('n_steps_latent_charged'),
    'Gated (kWh)': B(s)['Q_C_latent_kWh'],
    'Ungated (kWh)': B(s)['Q_C_latent_ungated_kWh'],
    'Charged w/ cooling OFF': B(s).get('latent_kWh_with_cooling_off'),
    'Charged while HEATING': B(s).get('latent_kWh_while_heating'),
    'Latent heating (kWh)': B(s)['Q_H_latent_kWh'],
} for s in results])
display(lat.style.format({'Gated (kWh)': '{:,.2f}', 'Ungated (kWh)': '{:,.2f}',
                          'Charged w/ cooling OFF': '{:.6f}',
                          'Charged while HEATING': '{:.6f}',
                          'Latent heating (kWh)': '{:.4f}'}).hide(axis='index'))

if monthly:
    names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    display(pd.DataFrame([monthly], columns=names, index=['kWh'])
            .style.format('{:.2f}'))
    print('Gated latent cooling by month at the canonical state — the load sits in')
    print('the austral summer, which is the southern-hemisphere phase check.')

## 7 · The chart

In [ ]:
subprocess.run([sys.executable, 'tools/diagnostics/make_closed_balance_chart.py',
                '--raw',    'results/au_canonical_essendon/trajectory_raw.json',
                '--outdir', 'results/au_canonical_essendon',
                '--stem',   'au_canonical_trajectory_essendon',
                '--title',  'Apt 305, 50 Barry St Carlton — the canonical trajectory '
                            'in methodology order, on the clean weather file',
                '--note',   'Literature corrections first (C1, C2), then the found '
                            'implementation defects, then the closure fixes. Every state '
                            'measured with the same closure-capable instrument. '
                            'Each metric on its own axis.'],
               check=True)

from IPython.display import Image, display
display(Image('results/au_canonical_essendon/au_canonical_trajectory_essendon.png'))

## 8 · The regression suite

The assertions behind everything above. Must be **green, exit 0, nothing
skipped** — a skipped worktree test is a test that ran no assertions, which is
easy to mistake for a pass if only the exit code is glanced at. The full refspec
fetched in section 1 is what keeps them from skipping.

In [ ]:
out = Path('results/au_canonical_essendon/pytest.txt')
env = dict(os.environ, PYTHONPATH=str(REPO / 'pybuildingenergy' / 'src'))

proc = subprocess.Popen([sys.executable, '-m', 'pytest', 'tests/', '-v', '-rs',
                         '--tb=short', '-p', 'no:cacheprovider'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)
lines = []
for line in proc.stdout:
    lines.append(line)
    if 'FAILED' in line or 'ERROR' in line or 'SKIPPED' in line or '=====' in line:
        print(line, end='')
proc.wait()

lines.append(f'\nexit code: {proc.returncode}\n')
out.write_text(''.join(lines))
print(''.join(lines[-3:]))
print('full log ->', out)
assert proc.returncode == 0, 'the regression suite is not green'
assert not any('SKIPPED' in l for l in lines), \
    'a test skipped — it executed no assertions; check the branch fetch in section 1'

## 9 · The wind diagnostic — closing the earlier verdict (c)

The same engine run **twice**, changing only the h_ce model
(`external_convection_model` / `window_convection_model` = `table` recovers the
ISO constant exactly). One switch, no worktrees, nothing else different — which
is what makes it a controlled experiment and what lets the cooling change be
attributed to bands of wind speed.

The question is whether the C2 effect is now explained by **real** wind:

* do cooling-plant-on hours coincide with above-pivot wind?
* how much of the C2 cooling delta comes from genuine, non-zero wind bands?

In [ ]:
proc = subprocess.Popen(
    [sys.executable, 'tools/diagnostics/wind_h_ce_diagnostic.py',
     '--weather', EPW,
     '--outdir', 'results/diagnostics',
     '--tag', 'essendon',
     '--expect-weather', 'Essendon',
     '--compare-to', 'results/diagnostics/wind_stats.json'],   # the RO run, for contrast
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:
    if 'it/s' in line or 'it]' in line:      # tqdm progress bars, not findings
        continue
    print(line, end='')
proc.wait()
print('\nexit code:', proc.returncode)

In [ ]:
from IPython.display import Image, display
display(Image('results/diagnostics/wind_distribution_essendon.png'))

In [ ]:
w = json.loads(Path('results/diagnostics/wind_stats_essendon.json').read_text())
s = w['summary']

print(f"VERDICT: ({w['verdict']})   —  replaces the earlier (c) on the RO file\n")
print(f"  {s['pct_extra_from_nonzero_wind']:.1f} % of the {s['delta_C']:+.2f} kWh comes "
      f"from genuine, non-zero wind bands")
print(f"  {s['pct_cooling_hours_above_pivot']:.1f} % of the {s['n_cooling_dyn']} "
      f"cooling-plant hours are above the 4 m/s pivot, against "
      f"{s['pct_hours_above_pivot']:.1f} % of the year")
print(f"  cooling-on mean wind {s['mean_wind_cooling_dyn']:.2f} m/s = "
      f"{s['wind_ratio_cooling_to_annual']:.2f}x the annual mean\n")

bands = pd.DataFrame(s['bands'])[['label', 'hours', 'extra_cooling_kWh', 'share_pct']]
bands.columns = ['Wind band', 'Hours', 'Extra sensible cooling (kWh)', 'Share']
display(bands.style.format({'Extra sensible cooling (kWh)': '{:+.2f}',
                            'Share': '{:.1f} %'}).hide(axis='index'))
print('Share is of the SIGNED total, so a band opposing the total reads negative')
print('and the four shares sum to 100 %.')

### The written verdict

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/diagnostics/wind_verdict_essendon.md').read_text()))

## 10 · The reports

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/au_canonical_essendon/comparison.md').read_text()))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/au_canonical_essendon/NOTES.md').read_text()))

### What the paper must be redone against

`SUPERSEDED.md` maps every previously-drafted number to its replacement — the
9.69 headline, the trajectory table, the C2 sign reversal, the six-state tables,
and `corrected_weather_results_rewrite.tex`. **No `.tex` was edited.**

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/au_canonical_essendon/SUPERSEDED.md').read_text()))

## 11 · Before quoting any of this

1. **The gate is the whole point.** No reduction percentage and no kWh/m²
   headline is final unless the V2 residual is under 5 % on **every** state, the
   ADJ transmission is in the inventory, the latent is gated, and the regression
   suite is green. Section 6 asserts all four; if that cell raises, nothing below
   it is quotable.

2. **The total moved only −7.1 %, but the components moved far more.** Heating
   +40.9 %, cooling −90.6 %. Quoting the total alone hides the fact that this is
   a different result, not a small correction to the old one.

3. **C2 changed sign.** Any text describing the wind-dependent h_ce as
   *increasing* cooling is now wrong, not merely imprecise. The earlier (c)
   verdict was a correct diagnosis of the *input*: `simplecombined` returns
   `4 + 4u` and reduces to the ISO constant at 4 m/s exactly as documented, on
   both files.

4. **The RO file is still in `weather_cache/`, deliberately.** The before/after
   contrast needs it. `CANONICAL_EPW` and the wind-column preflight are what stop
   it being picked up by accident — do not "tidy" either away.

5. **The six-state closed-balance harness was not re-run here.** If Tables 4/5
   come from it rather than from this trajectory, re-run it on the clean file
   before quoting — the command is in `SUPERSEDED.md` §4, and it needs
   `--closure-base 978db37` for the reason given in section 3.